[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://drive.google.com/file/d/1tUcZ_Fj0h3cd-pB_Tn0DYVpKwzxoAvIs/view?usp=sharing)

# Agent Evaluation – FloTorch Hosted Agent

This notebook demonstrates how to evaluate an agent deployed on the FloTorch gateway. Floeval sends each test case to the gateway, the hosted agent processes it, and Floeval captures the trace and scores the result.

**FloTorch Console:** [https://docs.flotorch.cloud/introduction/](https://docs.flotorch.cloud/introduction/)

**Prerequisites**
- Agent deployed in the [FloTorch Console](https://docs.flotorch.cloud/introduction/)
- `pip install floeval[flotorch]`
- FloTorch API key and gateway URL

**Objectives**
- Install Floeval with FloTorch support and configure credentials
- Load a partial agent dataset from a JSON file you provide
- Create a FloTorch runner with `create_flotorch_runner`
- Run `AgentEvaluation` with the runner and inspect results

## 1. Installation

Install Floeval with FloTorch support. Required for evaluating agents deployed on the FloTorch gateway.

In [ ]:
%pip install floeval[flotorch]>=0.2.0b1

## 2. Configuration Constants

Set the following constants before running. Obtain credentials and agent names from the [FloTorch Console](https://docs.flotorch.cloud/introduction/).

**Provider flexibility:** You can use any OpenAI-compatible provider. For FloTorch-hosted agents, use the FloTorch gateway URL and keys from the [FloTorch Console](https://docs.flotorch.cloud/introduction/).

In [ ]:
import getpass

# FloTorch gateway configuration
FLOTORCH_BASE_URL = "https://gateway.flotorch.cloud/openai/v1"
FLOTORCH_API_KEY = getpass.getpass("your-flotorch-api-key")
FLOTORCH_CHAT_MODEL =  "<flotorch-model>"
FLOTORCH_AGENT_NAME = "<agent-name>"
FLOTORCH_EMBEDDING_MODEL = "<flotorch-embedding-model>"

## 3. Imports

Import Floeval components for loading agent datasets, configuring providers, and creating a FloTorch runner.

In [ ]:
from pathlib import Path

from floeval.api.agent_evaluation import AgentEvaluation
from floeval.api.dataset_loaders.agent_file_loader import AgentDatasetLoader
from floeval.config.schemas.io.llm import OpenAIProviderConfig
from floeval.flotorch import create_flotorch_runner

## 4. Configure the LLM (FloTorch Gateway)

Build an OpenAI-compatible provider config that points to the FloTorch gateway.

In [ ]:
llm_config = OpenAIProviderConfig(
    base_url=FLOTORCH_BASE_URL,
    api_key=FLOTORCH_API_KEY,
    chat_model=FLOTORCH_CHAT_MODEL,
    embedding_model=FLOTORCH_EMBEDDING_MODEL,
)

## 5. Create the FloTorch Runner

`create_flotorch_runner(...)` connects to your deployed agent by name so Floeval can invoke it during evaluation.

### Create the FloTorch Runner

`create_flotorch_runner(agent_name, llm_config=...)` returns a gateway-backed runner that is later passed to `AgentEvaluation` via `agent_runner`.


In [ ]:
agent_name = FLOTORCH_AGENT_NAME
runner = create_flotorch_runner(agent_name, llm_config=llm_config)
print(f"Runner created for agent: {agent_name}")

## 6. Load the Partial Dataset (JSON)

Minimal JSON shape (partial agent rows):

```json
{
  "samples": [
    {
      "user_input": "...",
      "reference_outcome": "...",
      "reference_tool_calls": [{ "name": "...", "args": {} }]
    }
  ]
}
```
**Example file**  
<a href="../datasets/agent_evaluation/sample_agent_partial_flotorch.json" download="sample_agent_partial_flotorch.json">sample_agent_partial_flotorch.json</a>

Provide the dataset JSON path (or upload in Colab) and load it with `AgentDatasetLoader`.

In [ ]:
try:
    from google.colab import files
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if _IN_COLAB:
    print("Upload your agent dataset JSON file:")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded.")
    dataset_path = Path(next(iter(uploaded.keys())))
else:
    dataset_path = Path(input("Enter path to agent dataset JSON file: ").strip().strip('"')).expanduser()


In [ ]:
dataset = AgentDatasetLoader.from_file(dataset_path)
print(f"Partial dataset loaded from {dataset_path}: {len(dataset.samples)} sample(s)")


## 7. Create and Run the Evaluation

Pass `agent_runner=runner` into `AgentEvaluation`, then execute `evaluation.run()` to compute metrics.

In [ ]:
evaluation = AgentEvaluation(
    dataset=dataset,
    agent_runner=runner,
    llm_config=llm_config,
    metrics=["goal_achievement", "response_coherence", "ragas:tool_call_accuracy"],
    default_provider="builtin",
)


In [ ]:
results = evaluation.run()
print("Summary:", results.summary)


## Summary

This notebook demonstrated the evaluation of agents deployed on the FloTorch gateway.

The key components included:

1. **LLM Configuration**: The FloTorch gateway URL and API key were configured for the evaluation.
2. **FloTorch Runner**: A runner was created with `create_flotorch_runner(agent_name, llm_config)` to connect to the deployed agent.
3. **Partial Dataset**: A partial agent dataset was loaded from JSON.
4. **Evaluation Execution**: The `agent_runner` parameter was passed to `AgentEvaluation` for gateway-based execution.
5. **Metrics**: The `goal_achievement`, `response_coherence`, and `tool_call_accuracy` metrics were run.

This example showcases the workflow for evaluating FloTorch-hosted agents.